# 05 · MQTT: retained, Last Will e QoS

**Objetivo:** ver, com experimentos reais num broker, os três recursos que tornam o MQTT confiável para dados industriais.

| Recurso | Pergunta que ele responde |
|---|---|
| **Retained** | Quem conecta agora vê o valor atual, ou espera a próxima mudança? |
| **Last Will (LWT)** | Como saber que um dispositivo morreu, se ele não pode avisar? |
| **QoS + sessão persistente** | O que acontece com as mensagens enquanto o consumidor está fora? |

Precisa de um broker em `localhost:1883` (`docker compose up -d` na raiz).

In [1]:
# Configuração: deixa a raiz do repositório no caminho de import
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from common.notebook import load, run_in_background, require_broker
require_broker()
demo = load("05-mqtt-qos-retain-lwt/demo.py")

broker MQTT encontrado em localhost:1883


## 1. Retained: o "último valor conhecido"

Experimento: publicamos o nível uma vez **com** retain e uma vez **sem**, e só depois conectamos um assinante (imagine um dashboard que acabou de abrir).

In [2]:
r = demo.retained()
for topico, valor, flag in r["late_subscriber_got"]:
    print(f"recebeu {topico.split('/', 3)[-1]:<16} = {valor}   (flag retain = {flag})")

recebeu retained/level   = 52.3   (flag retain = True)


**Interpretação:** só o tópico `retained/level` chegou. O broker **guarda a última mensagem retida** de cada tópico e entrega para quem assinar depois. A mensagem sem retain foi entregue a quem estava conectado naquele instante (ninguém) e se perdeu.

**Regra prática:** retain para **estado** (nível, modo, online/offline), sem retain para **eventos** (botão apertado, peça produzida). Um evento retido seria "reprocessado" por todo cliente novo.

## 2. Last Will: o broker avisa pela vítima

O dispositivo registra, **no momento da conexão**, uma mensagem de testamento (`offline`). Depois simulamos uma queda de energia: fechamos o socket sem mandar `DISCONNECT`.

In [3]:
w = demo.last_will()
print("o observador recebeu, em ordem:", w["watcher_saw"])

o observador recebeu, em ordem: ['online', 'offline']


**Interpretação:** `online` foi publicado pelo próprio dispositivo. `offline` foi publicado **pelo broker**, que percebeu o socket morto e entregou o testamento. Como o will foi registrado com retain, qualquer painel que abrir depois também vê `offline`.

Sem LWT, a única forma de detectar um dispositivo morto seria "não recebi nada há X segundos", e o X sempre é escolhido errado.

## 3. QoS e sessão persistente

Experimento: um historiador assina com `clean_session=False` (sessão persistente) e sai do ar. Enquanto está fora, o dispositivo publica 5 mensagens em **QoS 0** e 5 em **QoS 1**. Depois o historiador volta com o mesmo `client_id`.

In [4]:
q = demo.qos_and_sessions()
recebidas = q["received_after_reconnect"]
print("entregues após reconectar:", recebidas)
print("QoS 0 recuperadas:", sum(m.startswith("qos0") for m in recebidas), "de 5")
print("QoS 1 recuperadas:", sum(m.startswith("qos1") for m in recebidas), "de 5")

entregues após reconectar: ['qos1-0', 'qos1-1', 'qos1-2', 'qos1-3', 'qos1-4']
QoS 0 recuperadas: 0 de 5
QoS 1 recuperadas: 5 de 5


**Interpretação**

* **QoS 0** ("no máximo uma vez"): o broker não guarda nada para quem está fora. As 5 se perderam.
* **QoS 1** ("pelo menos uma vez"): o broker enfileirou as 5 para a sessão persistente e entregou quando o historiador voltou.
* Isso só funciona com **o mesmo client_id e clean_session=False**. Com sessão limpa, o broker esquece a assinatura ao desconectar.

| QoS | Garantia | Custo | Uso típico |
|---|---|---|---|
| 0 | pode perder | menor | telemetria de alta frequência, onde a próxima amostra substitui a perdida |
| 1 | pode duplicar | PUBACK por mensagem | eventos, contadores, comandos (com tratamento de duplicata) |
| 2 | exatamente uma vez | 4 mensagens por publicação | raro em IIoT, caro demais |

## Resumo

* **Retain** = estado atual para quem chega; **LWT** = morte detectada pelo broker; **QoS 1 + sessão persistente** = nada se perde enquanto o consumidor está fora.
* São esses três recursos que o Sparkplug B (notebook 07) e os gateways (notebook 08) usam por baixo.